# ProductIQ — Complete AI Reference
## LangChain · RAG · CrewAI · MCP · Research Agent · Analytics · Simulation

Mirrors the production backend code in a self-contained notebook.
Egyptian retail context — bilingual Arabic/English output.

**Stack:**
- **LLM:** Groq `llama-3.3-70b-versatile` via `langchain-groq`
- **RAG:** FAISS + `sentence-transformers/all-MiniLM-L6-v2`
- **Multi-Agent:** CrewAI (notebook) / LangChain persona pattern (production)
- **Web Research:** Tavily MCP (`mcp` Python SDK over stdio)
- **Deterministic Engine:** Pandas analytics, elasticity-based simulation

**Architecture pattern:** Two-tier AI — deterministic tier always runs the math;
LLM tier adds natural-language reasoning on top. Every result carries an `engine`
provenance field: `"llm"`, `"deterministic"`, or `"deterministic+llm"`.

In [ ]:
# ─── 1. Install dependencies (run once) ───
!pip install -q langchain langchain-core langchain-groq langchain-community langchain-huggingface
!pip install -q faiss-cpu sentence-transformers crewai pandas python-dotenv mcp>=1.0.0

In [ ]:
import os, json, math, re, time, asyncio, difflib
from datetime import datetime, timedelta
from typing import List, Optional
from dataclasses import dataclass, field, asdict
from contextlib import AsyncExitStack
from urllib.parse import urlparse

from dotenv import load_dotenv
import pandas as pd
import numpy as np

# Load API keys from the backend .env file
load_dotenv('../backend/.env')
if not os.getenv('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = 'PASTE-YOUR-KEY-HERE'
if not os.getenv('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY'] = 'tvly-dev-PASTE-YOUR-KEY-HERE'

print('GROQ_API_KEY:', 'set' if os.getenv('GROQ_API_KEY') else 'MISSING')
print('TAVILY_API_KEY:', 'set' if os.getenv('TAVILY_API_KEY') else 'MISSING')

---
## Part 1: LangChain — Groq LLM Setup
Mirrors `backend/app/services/ai/llm.py` — lazy init, status tracking, invoke with deterministic fallback.

In [ ]:
# ─── Groq LLM (mirrors llm.py) ───
from langchain_groq import ChatGroq

_llm = None
_last_error = None
GROQ_MODEL = os.getenv('GROQ_MODEL', 'llama-3.3-70b-versatile')

def _init_llm():
    global _llm, _last_error
    if _llm is not None:
        return _llm
    if not os.getenv('GROQ_API_KEY'):
        _last_error = 'GROQ_API_KEY not configured'
        return None
    try:
        _llm = ChatGroq(model=GROQ_MODEL, temperature=0.2,
                        api_key=os.getenv('GROQ_API_KEY'))
        _last_error = None
        return _llm
    except Exception as e:
        _last_error = f'{type(e).__name__}: {e}'
        return None

def llm_status():
    llm = _init_llm()
    return {
        'configured': bool(os.getenv('GROQ_API_KEY')),
        'available': llm is not None,
        'last_error': _last_error,
        'model': GROQ_MODEL if os.getenv('GROQ_API_KEY') else None,
    }

def invoke_llm(prompt, *, temperature=0.2, fallback=None, parser=None):
    global _last_error
    llm = _init_llm()
    if llm is None:
        return {'content': fallback or 'LLM unavailable.',
                'engine': 'deterministic', 'error': _last_error}
    try:
        llm.temperature = temperature
        resp = llm.invoke(prompt)
        raw = resp.content.strip()
        if parser:
            parsed = parser(raw)
            if parsed is not None:
                return {'content': parsed, 'engine': 'llm', 'error': None}
            return {'content': raw, 'engine': 'deterministic+llm',
                    'error': 'JSON parse failed'}
        return {'content': raw, 'engine': 'llm', 'error': None}
    except Exception as e:
        _last_error = f'{type(e).__name__}: {e}'
        return {'content': fallback or 'LLM call failed.',
                'engine': 'deterministic', 'error': _last_error}

def extract_json_block(text):
    text = text.strip()
    if text.startswith('```json'):
        text = text[7:]
    if text.startswith('```'):
        text = text[3:]
    if text.endswith('```'):
        text = text[:-3]
    text = text.strip()
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None

print('LLM status:', llm_status()['available'])
if llm_status()['available']:
    print(_llm.invoke('Say hello in Arabic.').content)

---
## Part 2: LangChain — Structured Recommendations (LCEL Chain)
Mirrors `chains.py` `ai_recommendations()`. Uses `ChatPromptTemplate`, `JsonOutputParser`, and Pydantic models.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class Recommendation(BaseModel):
    product: str = Field(description='Product name')
    action: str = Field(description='Action: restock / discount / bundle / remove')
    reason: str = Field(description='Why this action')
    confidence: int = Field(description='Confidence 0-100')

class RetailAnalysis(BaseModel):
    summary: str = Field(description='One-paragraph executive summary')
    recommendations: List[Recommendation]

parser = JsonOutputParser(pydantic_object=RetailAnalysis)
lang = 'Arabic'

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an AI retail analyst for the Egyptian market. '
     'Analyze the store data and return structured recommendations. '
     'Use EGP currency. Respond in {language}.\n{format_instructions}'),
    ('human', 'Sales data:\n{sales_data}\n\nInventory data:\n{inventory_data}')
]).partial(format_instructions=parser.get_format_instructions())

# LCEL chain
chain = prompt | _init_llm() | parser

sample_sales = '''Product,Qty Sold,Revenue EGP,Cost EGP
Samsung Galaxy A56,45,269550,229500
iPhone 16 Pro,12,479880,384000
Xiaomi Redmi Note 14,78,140400,117000
Sony WH-1000XM6,8,119920,84000
Samsung Galaxy Tab S10,15,224850,172500'''

sample_inventory = '''Product,Stock,Cost EGP,Price EGP,Supplier
Samsung Galaxy A56,120,5100,5990,النور للتوريدات
iPhone 16 Pro,30,32000,39990,الجمعة للتكنولوجيا
Xiaomi Redmi Note 14,0,1500,1800,الإلكترونيات الحديثة
Sony WH-1000XM6,5,10500,14990,سوني مصر
Samsung Galaxy Tab S10,20,11500,14990,النور للتوريدات'''

result = chain.invoke({
    'sales_data': sample_sales,
    'inventory_data': sample_inventory,
    'language': lang
})

print('Executive Summary:')
print(result['summary'])
print('\nRecommendations:')
for r in result['recommendations']:
    print(f"  {r['product']}: {r['action']} — {r['reason']} [{r['confidence']}%]")

---
## Part 3: LangChain — CEO Report
Mirrors `chains.py` `ai_ceo_report()`. Weekly summary with scaffolded KPIs and action items.

In [ ]:
ceo_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are the AI analyst for an Egyptian electronics retailer. '
     'Write the weekly CEO report. Respond in {language}.\n{format_instructions}'),
    ('human', 'Store data:\n{summary}')
])

class CeoReport(BaseModel):
    summary_en: str = Field(description='Executive summary in English')
    summary_ar: str = Field(description='Same summary in Arabic')
    action_items_en: List[str] = Field(description='5 prioritized action items in English')
    action_items_ar: List[str] = Field(description='Same 5 items in Arabic')

ceo_parser = JsonOutputParser(pydantic_object=CeoReport)
ceo_chain = ceo_prompt.partial(format_instructions=ceo_parser.get_format_instructions()) | _init_llm() | ceo_parser

analytics_summary = '''Revenue (30d): 735,570 EGP (-5.2% vs prior)
Profit (30d): 98,450 EGP, Margin: 13.4%
Inventory turnover: 4.2x/year
Top sellers: Samsung Galaxy A56 (45u, 269,550 EGP); iPhone 16 Pro (12u, 479,880 EGP)
Slow movers: Sony WH-1000XM6 (60d no sale, 52,500 EGP tied)
Stock risk: Xiaomi Redmi Note 14 stock=0 [out]'''

ceo_result = ceo_chain.invoke({
    'summary': analytics_summary,
    'language': 'Arabic'
})

print('CEO Report:')
print('EN:', ceo_result['summary_en'])
print('\nAR:', ceo_result['summary_ar'])
print('\nActions:')
for i, (en, ar) in enumerate(zip(ceo_result['action_items_en'], ceo_result['action_items_ar']), 1):
    print(f'  {i}. {en}')

---
## Part 4: RAG Pipeline — FAISS + HuggingFace Embeddings
Mirrors the notebook's existing RAG section with product knowledge base retrieval.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

product_kb = '''Product: Samsung Galaxy A56
Category: Smartphones | Price: 5,990 EGP | Margin: ~15%
Target: Egyptian youth, students | Season: Back-to-school, Ramadan

Product: iPhone 16 Pro
Category: Smartphones | Price: 39,990 EGP | Margin: ~20%
Target: High-income professionals

Product: Xiaomi Redmi Note 14
Category: Smartphones | Price: 1,800 EGP | Margin: ~10-12%
Target: Budget-conscious buyers | Highest volume

Product: Sony WH-1000XM6
Category: Headphones | Price: 14,990 EGP | Margin: ~30%
Target: Audiophiles, travelers | Niche premium

Product: Samsung Galaxy Tab S10
Category: Tablets | Price: 14,990 EGP | Margin: ~23%
Target: Professionals, students, artists'''

docs = [Document(page_content=product_kb)]
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks = splitter.split_documents(docs)
print(f'{len(chunks)} chunks created')

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embeddings)
rag_chain = RetrievalQA.from_chain_type(
    llm=_init_llm(),
    retriever=vectorstore.as_retriever(search_kwargs={'k': 2})
)

query = 'Which product has the best margin for a small shop in Cairo to stock?'
print(f'\nQ: {query}')
print(f'A: {rag_chain.invoke({"query": query})["result"]}')

---
## Part 5: CrewAI — Multi-Agent Board Meeting
4 agents (CFO, Marketing, Inventory, CEO) debate a stocking decision. The production equivalent
in `crew.py` uses the same persona pattern via LangChain `invoke_llm()` calls.

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

crew_llm = LLM(model=f'groq/{GROQ_MODEL}', temperature=0.3)

ceo = Agent(role='CEO',
    goal='Make the final strategic stocking decision',
    backstory='You run an electronics retail chain in Cairo. You weigh market '
              'opportunity, brand positioning, and long-term growth.',
    llm=crew_llm, verbose=True)

cfo = Agent(role='CFO',
    goal='Evaluate the financial impact of stocking decisions',
    backstory='You are the finance chief. You care about margins, cash flow, '
              'and ROI. You hate inventory that ties up capital.',
    llm=crew_llm, verbose=True)

marketing = Agent(role='Marketing Director',
    goal='Assess market demand and brand potential',
    backstory='You track Egyptian social media trends, competitor moves, and '
              'customer sentiment. You know what sells in Cairo.',
    llm=crew_llm, verbose=True)

inventory_mgr = Agent(role='Inventory Manager',
    goal='Keep stock levels optimal and the warehouse efficient',
    backstory='You manage the Cairo warehouse. You know what is overstocked, '
              'which suppliers are reliable, and storage limits.',
    llm=crew_llm, verbose=True)

print('4 agents ready')

In [ ]:
product_to_analyze = '''Product: Samsung Galaxy A56
Cost: 5,100 EGP | Selling Price: 5,990 EGP | Margin: ~15%
Monthly Sales: 45 units | Current Stock: 120 units
Supplier: النور للتوريدات (reliable, 3-day delivery)
Competition: Xiaomi Redmi Note 14 at 1,800 EGP, iPhone 16 Pro at 39,990 EGP
Market Trend: Strong mid-range demand in Egypt
Seasonality: High during back-to-school and Ramadan'''

t1 = Task(
    description=f'Analyze this product FINANCIALLY:\n{product_to_analyze}\n'
                'Focus on margin adequacy, capital tied up, ROI, overstocking risk.',
    expected_output='Financial assessment with numbers and a clear verdict',
    agent=cfo)
t2 = Task(
    description=f'Analyze this product for MARKETING:\n{product_to_analyze}\n'
                'Focus on brand strength, demand trends, competitor positioning, sentiment.',
    expected_output='Marketing assessment with market insights and a verdict',
    agent=marketing)
t3 = Task(
    description=f'Analyze this product for INVENTORY:\n{product_to_analyze}\n'
                'Focus on stock turnover, storage, supplier reliability, reorder timing.',
    expected_output='Inventory assessment with stock recommendations',
    agent=inventory_mgr)
t4 = Task(
    description='Review the CFO, Marketing, and Inventory assessments. '
                'Make the FINAL DECISION: stock or not, and how many units. Give clear reasoning.',
    expected_output='Final decision with quantity and reasoning',
    agent=ceo)

crew = Crew(
    agents=[cfo, marketing, inventory_mgr, ceo],
    tasks=[t1, t2, t3, t4],
    process=Process.sequential,
    verbose=True)

print('=' * 60)
print('AI BOARD MEETING')
print('=' * 60)
result = crew.kickoff()
print('\n' + '=' * 60)
print('FINAL DECISION')
print('=' * 60)
print(result)

---
## Part 6: Production Multi-Agent Equivalent (LangChain persona pattern)
This is what `crew.py` actually uses — sequential LangChain `invoke_llm()` calls
with persona prompts. Same 4 roles, lighter dependencies, same demo result.

In [ ]:
def _agent_call(role, persona, task, context, prior_analyses=''):
    prior_section = ''
    if prior_analyses:
        prior_section = 'Other department heads have already weighed in:\n' + prior_analyses
    prompt = f'''You are the {role} at an Egyptian electronics retail company in Cairo.
{persona}

Analyze this product decision:
{context}

Your task: {task}

{prior_section}

Give your assessment in 3-4 sentences. Be specific with numbers (EGP).
End with a clear one-line recommendation starting with RECOMMENDATION:'''
    result = invoke_llm(prompt, fallback=f'[{role} offline] Unable to analyze. RECOMMENDATION: defer to CEO.')
    return result['content']

context = product_to_analyze
print(f'--- Product Context ---\n{context}\n')
cfo_text = _agent_call('CFO', 'You care about margins, cash flow, ROI, and capital efficiency.',
                       'Assess the financial viability: margin adequacy, capital tied in stock, ROI.', context)
print(f'\n--- CFO ---\n{cfo_text}')
mkt_text = _agent_call('Marketing Director',
    'You track Egyptian social media trends, competitor moves, and customer sentiment.',
    'Assess market demand, brand pull, competitor positioning in Egypt.', context)
print(f'\n--- Marketing ---\n{mkt_text}')
inv_text = _agent_call('Inventory Manager',
    'You manage the Cairo warehouse. You know turnover, supplier reliability, storage limits.',
    'Assess stock health: turnover, days-of-stock, supplier reliability, reorder urgency.', context)
print(f'\n--- Inventory ---\n{inv_text}')
prior = f'CFO: {cfo_text}\n\nMarketing: {mkt_text}\n\nInventory: {inv_text}'
ceo_text = _agent_call('CEO',
    'You balance growth, risk, and cash. You make the final call after hearing all departments.',
    'Review all three assessments and make the FINAL DECISION: stock or not, how many units, and why.',
    context, prior)
print(f'\n--- CEO (Final Decision) ---\n{ceo_text}')

---
## Part 7: MCP Client — Tavily Web Search
Mirrors `mcp_client.py`. Spawns `npx tavily-mcp` over stdio, manages session lifecycle,
TTL cache, token-bucket rate limiter, monthly budget.

In [ ]:
@dataclass
class SearchResult:
    title: str
    url: str
    description: str
    age: str | None = None
    domain: str = ''

    @classmethod
    def from_url(cls, title, url, description, age=None):
        return cls(title=title, url=url, description=description,
                   age=age, domain=urlparse(url).netloc)

@dataclass
class SearchResponse:
    available: bool
    results: list[SearchResult] = field(default_factory=list)
    cached: bool = False
    reason: str | None = None
    remaining_budget: int | None = None

TOOL_NAME = 'tavily_search'
BUCKET_CAPACITY = 8.0
BUCKET_REFILL = 0.2

class TavilySearchClient:
    def __init__(self):
        self._lock = asyncio.Lock()
        self._session = None
        self._exit_stack = None
        self._cache = {}
        self._month = datetime.now().strftime('%Y-%m')
        self._calls_this_month = 0
        self._tokens = BUCKET_CAPACITY
        self._bucket_last = time.monotonic()
        self._unavailable_reason = None
        if not os.getenv('TAVILY_API_KEY'):
            self._unavailable_reason = 'TAVILY_API_KEY not configured'
        elif not os.getenv('TAVILY_MCP_COMMAND', 'npx'):
            self._unavailable_reason = 'npx not found'

    @property
    def available(self):
        return self._unavailable_reason is None

    def status(self):
        return {'available': self.available,
                'reason': self._unavailable_reason,
                'calls_this_month': self._calls_this_month,
                'cached_queries': len(self._cache)}

    async def _teardown(self):
        if self._exit_stack is not None:
            try:
                await self._exit_stack.aclose()
            except Exception:
                pass
        self._exit_stack = None
        self._session = None

    async def _ensure_session(self):
        async with self._lock:
            if self._session is not None:
                return self._session
            from mcp import ClientSession, StdioServerParameters
            from mcp.client.stdio import stdio_client
            stack = AsyncExitStack()
            env = {**os.environ}
            params = StdioServerParameters(
                command=os.getenv('TAVILY_MCP_COMMAND', 'npx'),
                args=['-y', 'tavily-mcp'],
                env=env)
            read, write = await stack.enter_async_context(stdio_client(params))
            session = await stack.enter_async_context(
                ClientSession(read, write))
            await asyncio.wait_for(session.initialize(), timeout=20)
            self._exit_stack = stack
            self._session = session
            return session

    async def _call_tool(self, tool_args):
        try:
            session = await self._ensure_session()
            return await asyncio.wait_for(
                session.call_tool(TOOL_NAME, tool_args), timeout=20)
        except Exception:
            await self._teardown()
            session = await self._ensure_session()
            return await asyncio.wait_for(
                session.call_tool(TOOL_NAME, tool_args), timeout=20)

    def _cache_key(self, query, count, freshness):
        return f'{query.strip().lower()}|{count}|{freshness or ""}'

    def _cache_get(self, key):
        entry = self._cache.get(key)
        if not entry:
            return None
        ts, results = entry
        if (time.time() - ts) > 21600:  # 6h TTL
            del self._cache[key]
            return None
        return results

    def _cache_put(self, key, results):
        self._cache[key] = (time.time(), results)

    def _check_month(self):
        current = datetime.now().strftime('%Y-%m')
        if current != self._month:
            self._month = current
            self._calls_this_month = 0

    def _budget_ok(self):
        self._check_month()
        return self._calls_this_month < 900

    def _take_token(self):
        now = time.monotonic()
        elapsed = now - self._bucket_last
        self._bucket_last = now
        self._tokens = min(BUCKET_CAPACITY, self._tokens + elapsed * BUCKET_REFILL)
        if self._tokens >= 1.0:
            self._tokens -= 1.0
            return True
        return False

    def _parse_results(self, mcp_result):
        for item in getattr(mcp_result, 'content', []) or []:
            text = getattr(item, 'text', None)
            if not text:
                continue
            try:
                payload = json.loads(text)
                items = payload.get('results') or payload.get('data') or []
                out = []
                for it in items:
                    url = it.get('url', '')
                    if not url:
                        continue
                    out.append(SearchResult.from_url(
                        title=it.get('title', ''), url=url,
                        description=(it.get('content') or '')[:400],
                        age=it.get('published_date') or it.get('age')))
                if out:
                    return out
            except (json.JSONDecodeError, TypeError, AttributeError):
                pass
            blocks = re.split(r'\n\s*\n', text)
            current = {}
            out = []
            for block in blocks:
                for line in block.splitlines():
                    if line.startswith('Title:'):
                        if current.get('url'):
                            out.append(SearchResult.from_url(
                                current.get('title',''), current.get('url',''),
                                current.get('content','')[:400]))
                        current = {'title': line[6:].strip()}
                    elif line.startswith('URL:'):
                        current['url'] = line[4:].strip()
                    elif line.startswith('Content:'):
                        current['content'] = line[8:].strip()
                    elif current and 'content' in current:
                        current['content'] += ' ' + line.strip()
            if current.get('url'):
                out.append(SearchResult.from_url(
                    current.get('title',''), current.get('url',''),
                    current.get('content','')[:400]))
            if out:
                return out
        return []

    async def search(self, query, count=5, freshness=None):
        if not self.available:
            return SearchResponse(available=False, reason=self._unavailable_reason)
        key = self._cache_key(query, count, freshness)
        cached = self._cache_get(key)
        if cached is not None:
            return SearchResponse(available=True, results=cached, cached=True)
        if not self._budget_ok():
            return SearchResponse(available=False, reason='budget exhausted')
        if not self._take_token():
            return SearchResponse(available=False, reason='rate limited')
        tool_args = {'query': query, 'max_results': count}
        if freshness:
            tool_args['time_range'] = freshness
        try:
            result = await self._call_tool(tool_args)
        except Exception as e:
            return SearchResponse(
                available=False,
                reason=f'research call failed: {type(e).__name__}: {e}')
        self._calls_this_month += 1
        results = self._parse_results(result)
        self._cache_put(key, results)
        return SearchResponse(available=True, results=results, cached=False)

tavily = TavilySearchClient()
print('Tavily MCP client ready')
print('Available:', tavily.available)
if tavily.available:
    print('Status:', json.dumps(tavily.status(), indent=2))

---
## Part 8: Tavily MCP — Live Search Demo
Runs a real query through the Tavily MCP server (requires Node.js + npx).

In [ ]:
async def demo_search():
    if not tavily.available:
        print('Tavily unavailable:', tavily.status()['reason'])
        return
    resp = await tavily.search('Samsung Galaxy A56 price in Egypt EGP 2026', count=3)
    print(f'Available: {resp.available} | Cached: {resp.cached} | Results: {len(resp.results)}')
    for r in resp.results:
        print(f'  - {r.title}')
        print(f'    URL: {r.url}')
        print(f'    {r.description[:200]}')
        print()

await demo_search()

---
## Part 9: Research Agent — Full Pipeline
Mirrors `agent.py`. Pipeline: Plan (LLM queries) -> Search (Tavily MCP) -> Synthesize
(LLM report with source citations) -> Cross-reference (internal catalog prices).

In [ ]:
def _plan_queries(question, product_hint=None):
    prompt = f'''You are planning web research for an Egyptian retailer.
Question: {question}
Product context: {product_hint or 'general market'}
Return ONLY valid JSON: {{"queries": ["q1", "q2", "q3"]}}
Rules: 3-5 queries targeting current price in Egypt (EGP), competitors,
customer reviews, market trend, local availability.'''
    result = invoke_llm(prompt, parser=extract_json_block)
    if result['engine'] == 'llm' and isinstance(result.get('content'), dict):
        queries = result['content'].get('queries')
        if isinstance(queries, list) and queries:
            return [str(q) for q in queries[:5]]
    base = product_hint or question
    return [f'{base} price in Egypt EGP', f'{base} competitors Egypt market',
            f'{base} reviews customer feedback', f'{base} availability Egypt stores']

async def _run_searches(queries, count=3):
    notes = []
    if not tavily.available:
        return [], [tavily.status()['reason'] or 'unavailable']
    seen = set()
    results = []
    for q in queries:
        resp = await tavily.search(q, count=count)
        if not resp.available:
            notes.append(resp.reason or f'search failed: {q}')
            continue
        for r in resp.results:
            if r.url not in seen:
                seen.add(r.url)
                results.append(r)
    return results, notes

def _synthesize(question, results):
    sources = [{'n': i+1, 'url': r.url, 'title': r.title}
               for i, r in enumerate(results)]
    if not results:
        return {'summary_en': 'No reliable market data found.',
                'summary_ar': 'لم يتم العثور على بيانات موثوقة.',
                'price_landscape': [], 'competitors': [], 'sources': [],
                'confidence': 0, 'engine': 'deterministic'}
    snippets = '\n'.join(
        f'[{i+1}] {r.title} ({r.url})\n{r.description}'
        for i, r in enumerate(results))
    prompt = f'''You are a market research analyst for an Egyptian retailer.
Question: {question}
Web sources (cite every claim with [n]):
{snippets}
Return ONLY valid JSON:
{{"summary_en": "2-3 sentences with [n] citations",
  "summary_ar": "Arabic version, keep [n] citations",
  "price_landscape": [{{"point": "...", "source_n": 1}}],
  "competitors": [{{"name": "...", "note": "...", "source_n": 1}}],
  "recommended_action": "...", "confidence": 0-100}}
RULES: never invent a price. Every claim must have source_n.'''
    result = invoke_llm(prompt, parser=extract_json_block)
    if result['engine'] == 'llm' and isinstance(result.get('content'), dict):
        data = result['content']
        data['sources'] = sources
        data['engine'] = 'llm'
        return data
    return {'summary_en': f'Found {len(results)} sources. See citations below.',
            'summary_ar': f'تم العثور على {len(results)} مصادر.',
            'price_landscape': [{'point': f'{r.title}: {r.description[:150]}',
                                 'source_n': i+1} for i, r in enumerate(results[:3])],
            'competitors': [], 'sources': sources, 'confidence': 55,
            'engine': 'deterministic'}

async def research_report(product_name):
    queries = _plan_queries(f'Should I stock {product_name}?', product_name)
    results, notes = await _run_searches(queries)
    data = _synthesize(f'Should I stock {product_name} in Egypt?', results)
    return {'product': product_name, 'queries': queries, 'notes': notes,
            'summary_en': data.get('summary_en', ''),
            'summary_ar': data.get('summary_ar', ''),
            'price_landscape': data.get('price_landscape', []),
            'competitors': data.get('competitors', []),
            'recommended_action': data.get('recommended_action', ''),
            'sources': data.get('sources', []),
            'confidence': data.get('confidence', 0),
            'engine': data.get('engine', 'deterministic')}

# Run the research pipeline
report = await research_report('Samsung Galaxy A56')
print(f"Product: {report['product']}")
print(f"Engine: {report['engine']}")
print(f"EN: {report['summary_en'][:300]}")
print(f"AR: {report['summary_ar'][:300]}")
print(f"Queries: {report['queries']}")
print(f"Sources: {len(report['sources'])}")
print(f"Confidence: {report['confidence']}")

---
## Part 10: Deterministic Analytics Engine
Mirrors `engine.py`. Pandas-based KPI computation, top sellers, slow movers with
lost profit, stock risk, and product DNA (8-dimension scoring). This is the
foundation for all AI features — always runs, always available.

In [ ]:
# ─── Sample data (mirrors the store's DataFrames) ───
np.random.seed(42)
dates = pd.date_range('2026-01-01', '2026-07-27', freq='D')

products = pd.DataFrame({
    'product_id': ['P001', 'P002', 'P003', 'P004', 'P005'],
    'product_name': ['Samsung Galaxy A56', 'iPhone 16 Pro', 'Xiaomi Redmi Note 14',
                     'Sony WH-1000XM6', 'Samsung Galaxy Tab S10'],
    'product_name_ar': ['سامسونج جالاكسي A56', 'آيفون 16 برو', 'شاومي ريدمي نوت 14',
                        'سوني WH-1000XM6', 'سامسونج جالاكسي تاب S10'],
    'category': ['Smartphones', 'Smartphones', 'Smartphones', 'Audio', 'Tablets'],
    'category_ar': ['هواتف ذكية', 'هواتف ذكية', 'هواتف ذكية', 'صوتيات', 'أجهزة لوحية'],
    'unit_cost_egp': [5100.0, 32000.0, 1500.0, 10500.0, 11500.0],
    'selling_price_egp': [5990.0, 39990.0, 1800.0, 14990.0, 14990.0],
})

inventory = pd.DataFrame({
    'product_id': ['P001', 'P002', 'P003', 'P004', 'P005'],
    'current_stock': [120, 30, 0, 5, 20],
    'reorder_point': [20, 10, 50, 8, 15],
})

# Generate synthetic sales
rows = []
product_rates = {'P001': 1.5, 'P002': 0.4, 'P003': 2.6, 'P004': 0.3, 'P005': 0.5}
for pid, rate in product_rates.items():
    for d in dates:
        qty = max(0, int(np.random.poisson(rate)))
        if qty == 0:
            continue
        prod = products[products['product_id'] == pid].iloc[0]
        rows.append({'date': d, 'product_id': pid,
                     'quantity': qty, 'unit_price_egp': prod['selling_price_egp'],
                     'discount_egp': 0})
sales = pd.DataFrame(rows)

# ─── Analytics engine (from engine.py) ───
def compute_analytics():
    df = sales.merge(products[['product_id','product_name','product_name_ar',
                                'category','category_ar','unit_cost_egp']], on='product_id')
    df['revenue'] = df['quantity'] * df['unit_price_egp'] - df.get('discount_egp', 0)
    df['cost'] = df['quantity'] * df['unit_cost_egp']
    df['profit'] = df['revenue'] - df['cost']

    today = df['date'].max()
    last_30 = df[df['date'] > today - timedelta(days=30)]

    kpis = {
        'revenue': round(float(last_30['revenue'].sum())),
        'profit': round(float(last_30['profit'].sum())),
        'margin': round(float(last_30['profit'].sum()) / max(float(last_30['revenue'].sum()), 1) * 100, 1),
    }

    top = (last_30.groupby(['product_id','product_name','product_name_ar'])
           .agg(sold=('quantity','sum'), revenue=('revenue','sum'), profit=('profit','sum'))
           .reset_index().sort_values('revenue', ascending=False).head(5))
    top_sellers = [{'name': r.product_name, 'sold': int(r.sold),
                    'revenue': round(float(r.revenue))} for r in top.itertuples()]

    # Slow movers with lost profit
    last_sale = df.groupby('product_id')['date'].max()
    slow = []
    for _, inv in inventory.iterrows():
        pid = inv['product_id']
        ls = last_sale.get(pid)
        days_no_sale = (today - ls).days if ls is not None else 999
        if days_no_sale >= 14:
            prod = products[products['product_id'] == pid].iloc[0]
            tied = float(inv['current_stock'] * prod['unit_cost_egp'])
            margin_pct = ((prod['selling_price_egp'] - prod['unit_cost_egp'])
                          / max(prod['selling_price_egp'], 1)) * 100
            profit_data = df[df['product_id'] == pid]['profit']
            total_profit = float(profit_data.sum()) if not profit_data.empty else 0
            total_days = max(1, (today - df['date'].min()).days)
            monthly_profit = abs(total_profit) / (total_days / 30.0)
            lost_profit = round((days_no_sale / 30.0) * monthly_profit)
            slow.append({'name': prod['product_name'], 'days_no_sale': days_no_sale,
                        'stock': int(inv['current_stock']),
                        'tied_capital': round(tied),
                        'lost_profit_egp': lost_profit})
    slow_movers = sorted(slow, key=lambda x: -x['tied_capital'])[:5]

    return {'kpis': kpis, 'top_sellers': top_sellers, 'slow_movers': slow_movers}

a = compute_analytics()
print('KPIs:')
for k, v in a['kpis'].items():
    print(f'  {k}: {v}')
print('\nTop Sellers:')
for t in a['top_sellers']:
    print(f'  {t["name"]}: {t["sold"]} units, {t["revenue"]} EGP')
print('\nSlow Movers:')
for s in a['slow_movers']:
    print(f'  {s["name"]}: {s["days_no_sale"]}d no sale, '
          f'{s["tied_capital"]} EGP tied, {s["lost_profit_egp"]} EGP lost')

---
## Part 11: Product DNA — 8-Dimension Scoring
Mirrors `engine.py` `product_dna()`. Scores 0-100 across popularity, margin, demand,
risk, competitiveness, turnover, growth, and value.

In [ ]:
def product_dna(product_id):
    df = sales.merge(products[['product_id','product_name','product_name_ar',
                                'category','category_ar','unit_cost_egp','selling_price_egp']], on='product_id')
    df['revenue'] = df['quantity'] * df['unit_price_egp']
    today = df['date'].max()
    last_30 = df[df['date'] > today - timedelta(days=30)]
    prod = products[products['product_id'] == product_id].iloc[0]
    p30 = last_30[last_30['product_id'] == product_id]

    sold30 = float(p30['quantity'].sum())
    rev30 = float(p30['revenue'].sum())
    max_sold = max(float(last_30.groupby('product_id')['quantity'].sum().max()), 1)
    max_rev = max(float(last_30.groupby('product_id')['revenue'].sum().max()), 1)
    margin_pct = ((prod['selling_price_egp'] - prod['unit_cost_egp'])
                  / max(prod['selling_price_egp'], 1)) * 100
    cat_median = products[products['category'] == prod['category']]['selling_price_egp'].median()
    price_position = prod['selling_price_egp'] / max(cat_median, 1)
    velocity = sold30 / 30.0
    inv_row = inventory[inventory['product_id'] == product_id]
    stock = int(inv_row.iloc[0]['current_stock']) if not inv_row.empty else 0
    days_of_stock = stock / velocity if velocity > 0 else 999
    turnover_score = min(100, velocity / (max_sold / 30.0) * 100)

    dims = {
        'popularity': min(100, sold30 / max_sold * 100),
        'margin': min(100, margin_pct / 35 * 100),
        'demand': min(100, rev30 / max_rev * 100),
        'risk': min(100, max(0, 100 - days_of_stock / 90 * 100)) if sold30 > 0 else 20,
        'competitiveness': min(100, max(0, (2 - price_position) * 50)),
        'turnover': turnover_score,
    }
    dims = {k: int(min(100, max(0, v))) for k, v in dims.items()}
    health = int(np.mean(list(dims.values())))
    return {'product': {'id': product_id, 'name': prod['product_name']},
            'dimensions': dims, 'health_score': health}

dna = product_dna('P001')  # Samsung Galaxy A56
print(f"Product: {dna['product']['name']}")
print(f"Health Score: {dna['health_score']}")
for dim, score in dna['dimensions'].items():
    print(f'  {dim}: {score}')

---
## Part 12: What-If Simulation — Elasticity-Based Pricing Model
Mirrors `simulation.py`. Deterministic math computes demand change via category-specific
price elasticity, revenue/profit impact, margin changes, breakeven units, and profit decomposition.

In [ ]:
def elasticity_for(category, change_type):
    base = {'smartphones': 2.0, 'audio': 1.4, 'tablets': 1.6,
            'accessories': 2.4, 'wearables': 1.3, 'printers': 0.8, 'cameras': 0.9}
    cat = (category or '').lower().replace('s', '')
    e = base.get(cat, 1.5)
    if 'discount' in change_type or 'bundle' in change_type:
        return e * 1.2
    return e

def simulate_change(product_name, category, current_price, cost,
                    current_velocity_monthly, change_type, change_value_pct):
    v = abs(change_value_pct)
    e = elasticity_for(category, change_type)
    decrease = 'decrease' in change_type or 'discount' in change_type or 'bundle' in change_type
    demand_change = round(v * e, 1) if decrease else -round(v * e * 0.75, 1)
    new_price = current_price * (1 - v / 100) if 'decrease' in change_type or 'discount' in change_type else \
                current_price * (1 + v / 100) if 'increase' in change_type else current_price
    current_revenue = current_velocity_monthly * current_price
    new_revenue = current_velocity_monthly * (1 + demand_change / 100) * new_price
    revenue_impact = round(new_revenue - current_revenue)
    current_margin = current_price - cost
    new_margin = new_price - cost
    current_profit = current_velocity_monthly * current_margin
    new_profit = current_velocity_monthly * (1 + demand_change / 100) * new_margin
    profit_impact = round(new_profit - current_profit)
    current_margin_pct = round((current_margin / max(current_price, 1)) * 100, 1)
    projected_margin_pct = round((new_margin / max(new_price, 1)) * 100, 1)
    breakeven_units = max(0, math.ceil(current_profit / max(new_margin, 1)))
    volume_impact = round((demand_change / 100) * current_velocity_monthly * current_margin)
    margin_impact = round(current_velocity_monthly * (new_margin - current_margin))
    risk_level = 'high' if v > 15 else 'medium' if v > 7 else 'low'
    confidence = 65 if v > 15 else 75 if v > 7 else 85

    return {'product': product_name, 'current_price': current_price,
            'new_price': round(new_price), 'demand_change_pct': demand_change,
            'revenue_impact_egp': revenue_impact, 'profit_impact_egp': profit_impact,
            'current_margin_pct': current_margin_pct,
            'projected_margin_pct': projected_margin_pct,
            'breakeven_units': breakeven_units,
            'profit_breakdown': {'volume_impact_egp': volume_impact,
                                'margin_impact_egp': margin_impact},
            'risk_level': risk_level, 'confidence_pct': confidence}

sim = simulate_change(
    product_name='Samsung Galaxy A56', category='Smartphones',
    current_price=5990.0, cost=5100.0,
    current_velocity_monthly=45, change_type='price decrease',
    change_value_pct=10)

print(f"Scenario: {sim['product']} — 10% price decrease")
print(f"Current Price: {sim['current_price']} EGP -> New: {sim['new_price']} EGP")
print(f"Demand Change: {sim['demand_change_pct']}%")
print(f"Revenue Impact: {sim['revenue_impact_egp']:+,} EGP")
print(f"Profit Impact: {sim['profit_impact_egp']:+,} EGP")
print(f"Margin: {sim['current_margin_pct']}% -> {sim['projected_margin_pct']}%")
print(f"Breakeven: {sim['breakeven_units']} units/month")
print(f"Risk: {sim['risk_level']} | Confidence: {sim['confidence_pct']}%")

---
## Part 13: All AI API Endpoints — Map
Every AI feature in the backend is exposed via FastAPI in `routes.py`. Here is the complete map.

In [ ]:
endpoints = [
    ('GET', '/api/health', 'LLM status check — is Groq available?'),
    ('GET', '/api/analytics', 'Deterministic KPIs + slow-movers, auto-snapshot to memory'),
    ('GET', '/api/recommendations?lang=en', 'AI recommendations (LLM) or rule-based fallback'),
    ('GET', '/api/product-dna/{id}', '8-dimension scoring (always deterministic)'),
    ('POST', '/api/ceo-report?lang=en', 'Weekly CEO report (LLM or rule-based)'),
    ('POST', '/api/simulate', 'What-if pricing simulation (deterministic math)'),
    ('POST', '/api/board-meeting', '4-agent board meeting (LLM or persona fallback)'),
    ('GET', '/api/research/status', 'Tavily MCP availability and budget'),
    ('POST', '/api/research/chat', 'Conversational market research with history'),
    ('POST', '/api/research/report', 'Full pipeline: plan -> search -> synthesize -> cross-ref'),
    ('GET', '/api/memory/research-history', 'Past research reports from SQLite'),
    ('GET', '/api/memory/board-decisions', 'Past board meeting decisions'),
    ('GET', '/api/memory/metrics-snapshots', 'Historical KPI snapshots'),
]
print(f'AI Endpoints ({len(endpoints)}):')
for method, path, desc in endpoints:
    print(f'  {method:6s} {path:40s} {desc}')

---
## Summary: AI Components Reference

| Component | Location (backend) | Technology | Deterministic Fallback |
|---|---|---|---|
| **LLM** | `services/ai/llm.py` | Groq `llama-3.3-70b-versatile` via `langchain-groq` | Returns `{'engine': 'deterministic', 'error': ...}` |
| **Recommendations** | `services/ai/chains.py` | `ChatPromptTemplate` + `JsonOutputParser` + LCEL | Rule-based from analytics data |
| **CEO Report** | `services/ai/chains.py` | LangChain prompt + `JsonOutputParser` | Scaffolded KPIs + default action items |
| **Board Meeting** | `services/ai/crew.py` | Persona prompts via `invoke_llm()` (4 sequential agents) | Rule-based per-role responses |
| **Research Agent** | `services/research/agent.py` | Plan -> Tavily MCP -> Synthesize (LLM) | Keyword queries + raw snippet summary |
| **Tavily MCP** | `services/research/mcp_client.py` | `mcp.ClientSession` over stdio | Returns `available=False` with reason |
| **Analytics** | `services/analysis/engine.py` | Pandas (always deterministic) | N/A (always runs) |
| **Simulation** | `services/analysis/simulation.py` | Elasticity-based math (always deterministic) | N/A (always runs) |
| **Memory** | `services/memory/store.py` | SQLite | N/A (persistence layer) |
| **RAG (notebook only)** | `notebooks/ProductIQ_AI_Operations.ipynb` | FAISS + `all-MiniLM-L6-v2` | Keyword fallback |
| **CrewAI (notebook only)** | `notebooks/ProductIQ_AI_Operations.ipynb` | `crewai.Agent` + `Crew` | N/A (notebook demonstration) |

**Architecture pattern:** Two-tier AI — deterministic tier (Pandas + math) always runs;
LLM tier (Groq via LangChain) adds natural language on top. Every response carries
an `engine` field so the UI can show appropriate badges: AI-generated / Rule-based / Computed.